# Fenrir end-to-end demo

Este notebook enseña un flujo completo y utilizable de `Fenrir` sobre un dataset sintetico de clustering.
Importa primero la libreria instalada y, si no existe, usa el repo local sin depender de rutas absolutas.

Objetivos:
- ajustar el modelo y leer bien la mejor configuracion
- explotar leaderboard, metricas, perfiles de cluster y variables influyentes
- validar holdout, estabilidad y subconjuntos compactos
- exportar un workbook ejecutivo
- cerrar con una celda de validacion para comprobar que el flujo funciona correctamente

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

candidate_project_roots = [
    Path.cwd(),
    Path.cwd() / "fenrir",
    Path.cwd().parent,
    Path.cwd().parent / "fenrir",
]

try:
    import fenrir as fenrir_package
except ModuleNotFoundError:
    project_root = next(
        (
            path
            for path in candidate_project_roots
            if (path / "core.py").exists() and (path / "__init__.py").exists()
        ),
        None,
    )
    if project_root is None:
        raise RuntimeError(
            "No se ha encontrado fenrir instalado ni una copia local del proyecto."
        )

    PACKAGE_PARENT = project_root.parent
    if str(PACKAGE_PARENT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_PARENT))

    import fenrir as fenrir_package

fenrir_package = importlib.reload(fenrir_package)
Fenrir = fenrir_package.Fenrir
package_file = getattr(fenrir_package, "__file__", None)
if package_file is None:
    raise RuntimeError("No se pudo resolver la ruta del paquete fenrir cargado.")

PROJECT_ROOT = Path(package_file).resolve().parent

SEED = 13
print(f"Fenrir importado desde: {PROJECT_ROOT}")

In [ ]:
blobs = make_blobs(
    n_samples=96,
    centers=3,
    n_features=4,
    cluster_std=0.65,
    random_state=SEED,
    return_centers=False,
)
values = blobs[0]
labels = blobs[1]

df = pd.DataFrame(values, columns=['x1', 'x2', 'x3', 'x4'])
df['segmento'] = pd.Series(labels).map({0: 'rojo', 1: 'azul', 2: 'verde'})
df['canal'] = np.where(labels == 0, 'web', np.where(labels == 1, 'store', 'partner'))
df['ticket_medio'] = np.where(labels == 0, 'alto', np.where(labels == 1, 'medio', 'bajo'))
df['target'] = labels

df.head()

In [ ]:
model = Fenrir(
    df,
    target='target',
    variance_threshold=0.85,
    cluster_range=range(3, 5),
    algorithms=('kmeans', 'gmm'),
    scalers={'StandardScaler': StandardScaler()},
    random_state=SEED,
)
labels_pred = model.fit_predict()

best_configuration = model.best_configuration().rename('valor').to_frame()
leaderboard = model.summary(top_n=5)
metrics = model.metric_report()

display(best_configuration)
display(leaderboard)
display(metrics)

In [ ]:
source_feature_report = model.source_feature_report(top_n=6)
influential_variables = model.list_influential_variables(top_n=6)
cluster_sizes = model.cluster_size_report()
cluster_profile = model.cluster_groupby(top_n=4)
cluster_vs_target = model.cluster_target_report(normalize='index')
influential_dataset = model.influential_dataset(top_n=4, include_target=True)

display(source_feature_report)
display(cluster_sizes)
display(cluster_profile)
display(cluster_vs_target)
display(influential_dataset.head())
influential_variables

In [ ]:
holdout_summary = model.evaluate_holdout(test_size=0.25, n_splits=2)
holdout_top = model.holdout_report(top_n=3)
compact_search = model.search_influential_subsets(
    min_features=2,
    max_features=4,
    criterion='holdout_silhouette',
    include_holdout=True,
    test_size=0.25,
    n_splits=2,
)
best_compact_variables = model.best_influential_variable_list()
best_compact_model = model.best_influential_model()
stability_summary = model.evaluate_stability(sample_fraction=0.8, n_splits=2)
stability_top = model.stability_report(top_n=3)
analysis = model.analysis_report(
    top_n=5,
    leaderboard_top_n=5,
    holdout_top_n=3,
    compact_max_features=4,
    test_size=0.25,
    n_splits=2,
    include_stability=True,
    stability_n_splits=2,
    stability_top_n=3,
)

display(holdout_top)
display(compact_search.head(4))
display(stability_summary)
display(stability_top)
display(analysis['leaderboard'])
best_compact_variables

In [ ]:
export_dir = PROJECT_ROOT / 'examples' / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
export_path = model.export_executive_workbook(file_name=str(export_dir / 'fenrir_demo.xlsx'))
print(export_path)
Path(export_path).exists()

In [ ]:
assert len(labels_pred) == len(df)
assert hasattr(model, 'interactive_workbench')
assert not leaderboard.empty and 'configuration' in leaderboard.columns
assert {'metric_label', 'value_display'}.issubset(metrics.columns)
assert {'cluster_size', 'cluster_share', 'cluster_balance'}.issubset(cluster_sizes.columns)
assert {'target_dominante', 'share_dominante', 'lectura_target'}.issubset(cluster_vs_target.columns)
assert not holdout_top.empty and 'configuration' in holdout_top.columns
assert not stability_top.empty and 'configuration_status' in stability_top.columns
assert isinstance(best_compact_variables, list) and len(best_compact_variables) >= 2
assert best_compact_model.best_configuration().shape[0] > 0
assert 'leaderboard' in analysis and not analysis['leaderboard'].empty
assert Path(export_path).exists()
print('Notebook Fenrir validado correctamente.')